# TP Jour 2 — Modèle d'Exécution Spark, Jointures Distribuées & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze**, nettoyer les anomalies, typer fortement les colonnes (dates/timestamps, numériques) et créer vos colonnes métier dérivées.
3. Mettre en œuvre une **jointure distribuée optimisée** avec **Broadcast Hash Join** (`F.broadcast`) sur une table de référence (sans créer de produit cartésien N-M !).
4. Sauvegarder votre table analytique propre dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

## 1. Chargement de vos tables Bronze & Diagnostic Qualité

### Exercice 1.1 — Profiler les valeurs manquantes, doublons et plages de valeurs
1. Chargez votre (ou vos) table(s) Delta depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` par colonne clé ainsi que le nombre de doublons éventuels sur l'identifiant métier.

In [ ]:
# TODO : Charger votre table principale depuis `/lakehouse/bronze/...`
BRONZE_MAIN_PATH = "/lakehouse/bronze/..."  # À adapter à votre sujet
df_bronze = None

assert df_bronze is not None, "TODO : Chargez votre table Bronze principale dans `df_bronze`."

# TODO : Calculer et afficher le nombre de valeurs nulles sur vos colonnes principales
null_report = None
assert null_report is not None, "TODO : Calculez le diagnostic de valeurs nulles `null_report`."
null_report.show(truncate=False)

## 2. Nettoyage, Typage Fort & Jointure Optimisée (`BroadcastHashJoin`)

### Rappels d'architecture distribuée :
- **Attention aux jointures N-à-M :** Si vous joignez deux tables de détail partageant une clé non unique, pré-agrégez d'abord à la bonne granularité pour ne pas dupliquer les lignes et fausser vos sommes !
- **Broadcast Hash Join (`F.broadcast(petite_table)`) :** Lorsqu'une table de dimension / nomenclature fait quelques kilo-octets ou méga-octets, la diffuser sur chaque exécuteur supprime totalement l'étape coûteuse de **Shuffle** sur la grande table de faits.
  *(Si votre dataset Kaggle ne comporte qu'un seul fichier plat, construisez ou extrayez une table de référence/catégories dédiée pour pratiquer cette jointure et vérifier le plan physique !)*

### Exercice 2.1 — Nettoyer, enrichir par Broadcast Join et écrire `/lakehouse/silver/<table_silver>`
1. Supprimez les doublons (`dropDuplicates`), filtrez les valeurs techniquement impossibles (ex. durées ou montants négatifs) et convertissez explicitement vos dates (`F.to_timestamp` / `F.to_date`).
2. Créez au moins **2 nouvelles colonnes métier dérivées** utiles pour vos analyses des Jours 3 & 4 (ex. durée écoulée, écart, ratio, tranche horaire, mois `yyyy-MM`, indicateur binaire `0/1`...).
3. Réalisez une jointure avec `F.broadcast(...)` et affichez `.explain("simple")` pour prouver la présence de `BroadcastHashJoin`.
4. Écrivez la table résultante au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [ ]:
SILVER_MAIN_PATH = "/lakehouse/silver/..."  # TODO : Définir le chemin cible de votre table Silver principale

# TODO : Construire `df_silver` (nettoyage + typage + colonnes métier + Broadcast Join)
df_silver = None

assert df_silver is not None, "TODO : Construisez le DataFrame nettoyé `df_silver`."
df_silver.explain("simple")

# TODO : Sauvegarder `df_silver` au format "delta" (mode overwrite + overwriteSchema=true) dans `SILVER_MAIN_PATH`
...

print(f"Table Silver écrite dans {SILVER_MAIN_PATH} : {spark.read.format('delta').load(SILVER_MAIN_PATH).count():,} lignes")

## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [ ]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# 2. TODO : Tester le Schema Enforcement avec un DataFrame au mauvais schéma dans un bloc try/except
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [ ]:
# TODO : Exécuter un MERGE INTO sur `SILVER_MAIN_PATH` puis comparer versionAsOf=0 et la version courante
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
...

history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)
assert history_df.count() >= 2, "TODO : Exécutez au moins une opération MERGE/UPDATE pour créer une version >= 1 !"